# Target B — Precision-first court citations via statute-overlap

**Strategy:** for each query, the law pipeline already predicts ~13 statutes (at F1≈0.7 law-only). The court paragraphs in the corpus that cite MANY of those statutes are high-precision court-citation candidates. Predict only a small number per query (K=2-5) where confidence is highest.

**Why this works when prior rerankers didn't:**
- No ML model. Pure regex statute extraction + set overlap.
- Precision is high by construction: a court paragraph that cites 4+ of the query's statutes is almost certainly on-topic.
- T4-compatible (CPU only, no GPU at inference).
- Generalizes — same procedure on val and test.

**Expected outcome:** joint F1 lift from 0.586 (law-only) to **0.61-0.66 (law + Target B court)**.

## Cell 1 — Setup

In [ ]:
import re, json, time, gc
import pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive')

SWISS_LAW_DIR = Path('/content/drive/MyDrive/swiss_law')
LAW_PIPE_DIR  = Path('/content/drive/MyDrive/Omnilex-Agentic-Retrieval-Competition')
DATA_DIR  = SWISS_LAW_DIR / 'data'
OUT_DIR   = SWISS_LAW_DIR / 'court_target_b_2026-05-22'
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Cell 2 — Load val, test, court corpus, law predictions

In [ ]:
val  = pd.read_csv(DATA_DIR / 'val.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
QID, QCOL, GOLDCOL = 'query_id', 'query', 'gold_citations'
val['gold_list'] = val[GOLDCOL].apply(lambda s: [c.strip() for c in re.split(r'[;,]', str(s)) if c.strip()])

# Law predictions (Option A — JSON saved by patched law notebook)
LAW_PREDS_VAL  = LAW_PIPE_DIR / 'retrieval/pipeline_output_v12/law_predictions_per_query.json'
LAW_PREDS_TEST = LAW_PIPE_DIR / 'retrieval/pipeline_output_v12/law_predictions_per_query_TEST.json'  # if you run law pipeline on test

law_preds_val  = json.loads(LAW_PREDS_VAL.read_text())  if LAW_PREDS_VAL.exists()  else {}
law_preds_test = json.loads(LAW_PREDS_TEST.read_text()) if LAW_PREDS_TEST.exists() else {}

print(f'Law preds (val):  {len(law_preds_val)} queries')
print(f'Law preds (test): {len(law_preds_test)} queries')
if not law_preds_test:
    print('NOTE: law predictions for test.csv not found.')
    print('       Run your law pipeline on test.csv, save as law_predictions_per_query_TEST.json.')
    print('       For now, this notebook will only evaluate on val.')

t0 = time.time()
court = pd.read_csv(DATA_DIR / 'court_considerations.csv', low_memory=False)
court['text'] = court['text'].astype(str)
print(f'Court corpus loaded: {len(court)} rows in {time.time()-t0:.1f}s')

## Cell 3 — Filter to federal-doctrinal pool

Hard filters: federal court only (BGE/ATF/DTF + docket like 1B_/4A_/5A_/6B_/7B_/8C_/9C_/2C_), no boilerplate, length 300-3000 chars (typical doctrinal range).

In [ ]:
FEDERAL_RE = re.compile(r'^(BGE|ATF|DTF)\s+\d+|^\d[A-Z]_\d+/\d+', re.I)
DISPOSITIF_RE = re.compile(
    r'^\s*(?:\d+\.?\s+)?'
    r'(?:Die Beschwerde wird|Le recours est|Il ricorso \u00e8|Im Namen|Au nom de|'
    r'Gerichtskosten\s|Les frais judiciaires|Es werden keine Kosten|'
    r'Il n\'est pas per\u00e7u|Lausanne,)', re.I)

mask = (
    court['citation'].str.match(FEDERAL_RE, na=False)
    & (~court['text'].str.match(DISPOSITIF_RE, na=False))
    & (court['text'].str.len().between(300, 3000))
)
pool = court[mask].reset_index(drop=True)
print(f'Federal-doctrinal pool: {len(pool)} / {len(court)} ({100*len(pool)/len(court):.1f}%)')

# Survival check on val court gold
all_court_gold = set()
for L in val['gold_list']: all_court_gold.update(L)
court_cit_set = set(court['citation'])
court_gold_in_corpus = all_court_gold & court_cit_set
court_gold_in_pool   = all_court_gold & set(pool['citation'])
print(f'Court gold in corpus: {len(court_gold_in_corpus)}/{len(all_court_gold)} (val gold incl. law)')
print(f'Court gold in filtered pool: {len(court_gold_in_pool)} / {len(court_gold_in_corpus)} survived filter')

del court; gc.collect()

## Cell 4 — Build statute index over pool (case-normalized)

Per pool doc: extract all statutes it cites. Store as `(full_spec_set, art_code_set)`. Case-normalize codes (StPO/STPO/stpo → stpo).

In [ ]:
STATUTE_RE = re.compile(
    r'\b[Aa]rt(?:icle|icolo|\.)?\.?\s+(\d+[a-z]?)'
    r'(?:\s+(?:Abs|al|cpv|para|paragraph)\.?\s+(\d+))?'
    r'(?:\s+(?:lit|let|lett)\.?\s+([a-z]))?'
    r'\s+([A-Z][A-Za-z]{1,7}\d?)\b'
)

STATUTE_ALIASES = {
    'stpo':['cpp'],   'cpp':['stpo'],
    'stgb':['cp'],    'cp':['stgb'],
    'zgb':['cc'],     'cc':['zgb'],
    'or':['co'],      'co':['or'],
    'zpo':['cpc'],    'cpc':['zpo'],
    'bgg':['ltf'],    'ltf':['bgg'],
    'bv':['cst','cost'], 'cst':['bv','cost'], 'cost':['bv','cst'],
    'atsg':['lpga'],  'lpga':['atsg'],
    'ivg':['lai'],    'lai':['ivg'],
    'uvg':['laa'],    'laa':['uvg'],
    'schkg':['lp'],   'lp':['schkg'],
    'emrk':['cedh'],  'cedh':['emrk'],
}

def parse_st(s):
    m = STATUTE_RE.search(s)
    if not m: return None
    art, abs_, lit, code = m.group(1), m.group(2), m.group(3), m.group(4).lower()
    full = f'art. {art}'
    if abs_: full += f' abs. {abs_}'
    if lit:  full += f' lit. {lit}'
    full += f' {code}'
    return full, f'art. {art} {code}', code

def aliases(art_code_lower):
    m = re.match(r'(art\.\s+\d+[a-z]?)\s+(\S+)', art_code_lower)
    if not m: return [art_code_lower]
    art, code = m.group(1), m.group(2)
    out = [art_code_lower]
    for alias in STATUTE_ALIASES.get(code, []):
        out.append(f'{art} {alias}')
    return out

# Cache file for the pool statute index
POOL_IDX_PATH = OUT_DIR / 'pool_statutes.npz'
if POOL_IDX_PATH.exists():
    data = np.load(POOL_IDX_PATH, allow_pickle=True)
    doc_full_set = data['full'].tolist()
    doc_ac_set   = data['ac'].tolist()
    saved_n = data['n_pool'].item()
    if saved_n != len(pool):
        print(f'WARN: cached pool size {saved_n} != current {len(pool)}; rebuilding')
        POOL_IDX_PATH.unlink()
if not POOL_IDX_PATH.exists():
    print('Indexing pool statutes (case-normalized)...')
    doc_full_set = [None]*len(pool)
    doc_ac_set   = [None]*len(pool)
    t0 = time.time()
    for i in range(len(pool)):
        text = pool['text'].iloc[i][:3000]
        fulls, acs = set(), set()
        for m in STATUTE_RE.finditer(text):
            art, abs_, lit, code = m.group(1), m.group(2), m.group(3), m.group(4).lower()
            acs.add(f'art. {art} {code}')
            full = f'art. {art}'
            if abs_: full += f' abs. {abs_}'
            if lit:  full += f' lit. {lit}'
            full += f' {code}'
            fulls.add(full)
        doc_full_set[i] = fulls
        doc_ac_set[i]   = acs
        if (i+1) % 200000 == 0:
            print(f'  {i+1}/{len(pool)} ({(time.time()-t0)/60:.1f} min)')
    print(f'  done in {(time.time()-t0)/60:.1f} min')
    np.savez(POOL_IDX_PATH, full=np.array(doc_full_set, dtype=object), ac=np.array(doc_ac_set, dtype=object), n_pool=len(pool))
    print(f'  cached to {POOL_IDX_PATH.name}')
print(f'doc_full_set[0]: {sorted(doc_full_set[0])[:5]}')
print(f'doc_ac_set[0]:   {sorted(doc_ac_set[0])[:5]}')

## Cell 5 — Score-and-predict function (with K and MIN_AC_MATCHES sweep)

Per query: score = 3·|full_overlap| + 1·|ac_overlap|. Predict only docs where ac_overlap ≥ MIN_AC.

In [ ]:
def query_statutes(law_preds_list):
    full_set, ac_set = set(), set()
    for s in law_preds_list:
        p = parse_st(s)
        if p:
            full_set.add(p[0])
            for ac in aliases(p[1]):
                ac_set.add(ac)
    return full_set, ac_set

def predict_court(qid, law_list, K, min_ac, min_full=1):
    """Return list of top-K court citations by statute-overlap score, with thresholds."""
    full_q, ac_q = query_statutes(law_list)
    if not ac_q: return []
    scored = []
    for i in range(len(pool)):
        ac_overlap = len(doc_ac_set[i] & ac_q)
        if ac_overlap < min_ac: continue
        full_overlap = len(doc_full_set[i] & full_q)
        if full_overlap < min_full: continue
        scored.append((i, 3*full_overlap + ac_overlap))
    scored.sort(key=lambda x: -x[1])
    return [pool['citation'].iloc[i] for i, _ in scored[:K]]

def joint_f1(law_pred_dict, court_pred_dict, gold_dict):
    per_q = []
    for qid, gold in gold_dict.items():
        gold_set = set(gold)
        if not gold_set: continue
        pred = set(law_pred_dict.get(qid, [])) | set(court_pred_dict.get(qid, []))
        tp = len(pred & gold_set)
        P = tp / max(1, len(pred))
        R = tp / len(gold_set)
        F = 2*P*R/(P+R) if (P+R) > 0 else 0.0
        per_q.append((qid, len(gold_set), len(pred), tp, P, R, F))
    return per_q

## Cell 6 — Sweep K × MIN_AC on val to find best (K, MIN_AC) combination

In [ ]:
val_gold = {r[QID]: r['gold_list'] for _, r in val.iterrows()}

# Law-only baseline
baseline = joint_f1(law_preds_val, {}, val_gold)
macro_baseline = np.mean([row[6] for row in baseline])
print(f'BASELINE (law-only): macro joint F1 = {macro_baseline:.3f}')

# Sweep K and MIN_AC
print(f"\n{'K':>3} {'MIN_AC':>7} | {'macro F1':>10} | {'\u0394 vs baseline':>14}")
best = (None, macro_baseline)
all_results = {}
for min_ac in [2, 3, 4, 5]:
    for K in [1, 2, 3, 4, 5, 7, 10]:
        court_preds = {qid: predict_court(qid, law_preds_val.get(qid, []), K, min_ac) for qid in val_gold}
        rows = joint_f1(law_preds_val, court_preds, val_gold)
        macro = np.mean([row[6] for row in rows])
        lift = macro - macro_baseline
        marker = '  \u2190 best' if macro > best[1] else ''
        if macro > best[1]: best = ((K, min_ac), macro)
        all_results[(K, min_ac)] = (macro, lift)
        print(f'{K:>3} {min_ac:>7} | {macro:>10.3f} | {lift:>+14.3f}{marker}')

print(f'\nBest: K={best[0][0]}, MIN_AC={best[0][1]}, macro F1 = {best[1]:.3f}')
BEST_K, BEST_MIN_AC = best[0]

## Cell 7 — Per-query detail with best (K, MIN_AC)

In [ ]:
court_preds_val = {qid: predict_court(qid, law_preds_val.get(qid, []), BEST_K, BEST_MIN_AC) for qid in val_gold}
rows = joint_f1(law_preds_val, court_preds_val, val_gold)

df = pd.DataFrame(rows, columns=['qid','gold','pred','tp','P','R','F1'])
df.to_csv(OUT_DIR / 'val_per_query.csv', index=False)
print(df.to_string(index=False))
macro = df[['P','R','F1']].mean()
print(f'\n=== MACRO JOINT (law + court): P={macro["P"]:.3f} R={macro["R"]:.3f} F1={macro["F1"]:.3f} ===')
print(f'    Lift vs law-only baseline: {macro["F1"] - macro_baseline:+.3f}')

# Per-query court contribution audit
print('\n--- Court contribution per query ---')
for qid in val_gold:
    gold = set(val_gold[qid])
    court_pred = set(court_preds_val.get(qid, []))
    court_tp = court_pred & gold
    court_fp = court_pred - gold
    print(f'  {qid}: {len(court_pred)} court preds | court TP = {len(court_tp)}, FP = {len(court_fp)}')
    for c in sorted(court_tp):
        print(f'    \u2713 {c}')
    for c in sorted(court_fp):
        print(f'    \u2717 {c}')

## Cell 8 — Generate test submission.csv

Requires `law_predictions_per_query_TEST.json`. If you don't have it yet, run your law pipeline on `test.csv` first.

In [ ]:
if not law_preds_test:
    print('SKIPPING: no test law predictions found.')
    print('Run your law pipeline on test.csv first, save as law_predictions_per_query_TEST.json next to the val one.')
else:
    court_preds_test = {}
    for _, r in test.iterrows():
        qid = r[QID]
        court_preds_test[qid] = predict_court(qid, law_preds_test.get(qid, []), BEST_K, BEST_MIN_AC)

    rows = []
    for _, r in test.iterrows():
        qid = r[QID]
        joint = list(dict.fromkeys(law_preds_test.get(qid, []) + court_preds_test.get(qid, [])))  # preserve order, dedupe
        rows.append({'query_id': qid, 'predicted_citations': ';'.join(joint)})
    sub = pd.DataFrame(rows)
    sub_path = OUT_DIR / 'submission.csv'
    sub.to_csv(sub_path, index=False)
    print(f'Submission written: {sub_path}')
    print(f'  {len(sub)} rows | avg citations per query = {sub["predicted_citations"].str.split(";").str.len().mean():.1f}')
    print(sub.head(3).to_string(index=False))

## Cell 9 — Cache court predictions for inspection / merging downstream

In [ ]:
# Save the court predictions per query + best (K, MIN_AC) config for reuse
config_path = OUT_DIR / 'target_b_config_and_preds.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump({
        'best_K': BEST_K,
        'best_MIN_AC': BEST_MIN_AC,
        'macro_baseline_law_only': macro_baseline,
        'macro_best_law_plus_court_b': float(macro['F1']),
        'court_preds_val':  court_preds_val,
        'court_preds_test': locals().get('court_preds_test', {}),
    }, f, ensure_ascii=False, indent=2)
print(f'Saved {config_path}')